# Height Error Analysis
Compares detected vs actual height across test subjects and visualises error.

**v11 results: 3 subjects tested**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Test data — update with your own measurements
data = {
    'Subject':        ['Person 1', 'Person 2', 'Person 3'],
    'Actual_Height':  [160.0, 173.0, 189.0],
    'Detected_Height':[160.5, 173.9, 189.2],
    'Actual_MUAC':    [30.0, 33.0, 37.0],
    'Detected_MUAC':  [29.2, 31.8, 35.6],
    'Actual_HC':      [54.5, 56.5, 57.0],
    'Detected_HC':    [45.4, 45.9, 46.1],
}

df = pd.DataFrame(data)
df['Height_Error'] = df['Detected_Height'] - df['Actual_Height']
df['MUAC_Error']   = df['Detected_MUAC']   - df['Actual_MUAC']
df['HC_Error']     = df['Detected_HC']     - df['Actual_HC']

print(df[['Subject','Height_Error','MUAC_Error','HC_Error']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Camera-Based Anthropometry v11 — Error Analysis', fontsize=14, fontweight='bold')

measurements = [
    ('Height', df['Actual_Height'], df['Detected_Height'], df['Height_Error'], 'steelblue', '±6 cm'),
    ('MUAC',   df['Actual_MUAC'],   df['Detected_MUAC'],   df['MUAC_Error'],   'coral',     '±3 cm'),
    ('HC',     df['Actual_HC'],     df['Detected_HC'],     df['HC_Error'],     'mediumpurple','±4 cm'),
]

for ax, (name, actual, detected, error, col, target) in zip(axes, measurements):
    x = np.arange(len(df))
    ax.bar(x - 0.2, actual,   0.38, label='Actual',   color='lightgray', edgecolor='gray')
    ax.bar(x + 0.2, detected, 0.38, label='Detected', color=col, alpha=0.8, edgecolor='gray')
    ax.set_xticks(x)
    ax.set_xticklabels(df['Subject'], rotation=15)
    ax.set_title(f'{name}  (target {target})')
    ax.set_ylabel('cm')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    # Error annotations
    for i, (xi, err) in enumerate(zip(x, error)):
        col2 = 'red' if abs(err) > float(target.replace('±','').replace(' cm','')) else 'green'
        ax.annotate(f'{err:+.1f}', (xi + 0.2, detected.iloc[i] + 0.3),
                    ha='center', fontsize=8, color=col2, fontweight='bold')

plt.tight_layout()
plt.savefig('../results/error_comparison_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/error_comparison_chart.png')

In [ ]:
# Summary statistics
print('=== Error Summary ===')
for col, target in [('Height_Error', 6), ('MUAC_Error', 3), ('HC_Error', 4)]:
    name   = col.replace('_Error','')
    mean_e = df[col].mean()
    max_e  = df[col].abs().max()
    within = (df[col].abs() <= target).all()
    status = '✅ PASS' if within else '❌ FAIL'
    print(f'  {name:8s}  mean={mean_e:+.1f}cm  max_abs={max_e:.1f}cm  target=±{target}cm  {status}')